# 09_04 With and without the passages, and where this goes in real work

*📖 Not in the book*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → numbers → model reads → model learns → model writes → MCP tool
> ```
>
> You are here: **after the main course**: the exam, with the book closed and open
>
> Project files this connects to:
>
> - `scripts/benchmark_grounding.py`
> - `src/homer_ai/retrieval.py`
> - `data/eval/questions.jsonl`
>
> **This notebook:** Does handing a model the passages make its answers better? We ask the same questions with and without them, and then look at what changes when the text is a company's own.
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- How the passages get into the prompt, and what the model is told to do with them
- How to compare a model with and without them, and what a small run showed
- Why the answer step can't be better than the search step
- What changes when the text is a company's own documents

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **headless mode** | running a chat program from a script, with no window: text in, text out | `claude -p "…"` |
| **hybrid search** | running a word search and a vector search and merging their results | BM25 + a sentence model, as in 09_02 |
| **reranking** | fetching many passages cheaply, then letting a stronger model put them in a better order | fetch 50, keep the best 5 |

## 3. The idea

### ❓ We have a search and a tool. Does any of it help?

That's the question to ask of everything you build, and the only way to answer it is the one from chapter 5: run it both ways and compare.

So we give the same questions to the same model twice. Once it gets only the question: the closed-book exam. Once it gets the question and the five passages our search found: the open-book exam. Then we score both.

I'll tell you now that the result isn't flattering for our search, and that's what makes it worth looking at.

## 4. Step by step

Each step is one question. Read it, guess what the code will do, run it, then look at what you got.

### ❓ Step 1: how do the passages reach the model?

As text in the prompt. There's no special channel. We paste the passages under the question, each with its reference in front, and add instructions: answer only from these, say so if they don't hold the answer, and end with the reference you used.

(With the MCP tool of [09_03](09_03_the_search_tool.ipynb) the passages arrive as a tool result. For the model it comes to the same thing: text that sits in front of it while it answers.)

Here is the prompt for one question, with the passages shortened so it fits on a screen.

In [ ]:
from homer_ai.retrieval import load_questions, search

questions = load_questions()
question = questions[25]  # "Which goddess does Diomedes wound in the hand?"
found = search(question["question"], k=5)
passages = "\n\n".join(f"[{p.ref}] {p.text[:150]}…" for p, _score in found)

prompt = (
    f"Question: {question['question']}\n\n"
    "Here are passages from Samuel Butler's translation. Each starts with its "
    "reference in square brackets: poem book:line.\n\n"
    f"{passages}\n\n"
    "Answer only from these passages. If they do not hold the answer, say so. "
    "Then end with one line in exactly this form, giving the reference of the "
    "passage your answer comes from:\n"
    "SOURCE: <odyssey or iliad> <book>:<line>"
)
print(prompt)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

A question, five passages with their references, and three instructions. The third passage, `[iliad 5:29]`, is the one that holds the answer (Butler's Venus is Aphrodite).

The last line, `SOURCE: …`, is there for scoring. A fixed form at the end is easy for a program to find, so we don't need a person, or another model, to read every answer.

</details>

### ❓ Step 2: how do I run the comparison?

With `scripts/benchmark_grounding.py`. For each question it calls Claude twice through the Claude Code command line in **headless mode** (`claude -p`: text in, text out, no window), with all tools switched off and a fixed system prompt, from an empty folder. So Claude sees only what we send.

```
uv run python scripts/benchmark_grounding.py --limit 8 --model sonnet
```

| Run | What Claude gets | What it must end with |
|---|---|---|
| plain | the question | `SOURCE: iliad 5` (poem and book) |
| grounded | the question + the 5 passages `search` returns | `SOURCE: iliad 5:29` (poem, book and line) |

And two scores:

- **right book**: the poem and book are one of the labelled places.
- **right passage** (grounded only): the cited line is inside a labelled place.

The plain run is only asked for the book, and that's deliberate. Our line numbers are lines of our own files, which no model has seen. Asking for them from memory would be unfair.

The script isn't part of the course's automatic checks: it needs Claude Code installed and uses your account. `--limit 8` takes 8 questions spread evenly over the 40. You don't have to run it to follow the rest.

### ❓ Step 3: what came out?

This is one recorded run, not a number to quote:

| | |
|---|---|
| Date | 24 November 2026 |
| Model | `sonnet` (the alias, through Claude Code 2.1.316) |
| Questions | 8 of the 40 (ids 1, 6, 11, 16, 21, 26, 31, 36) |
| Passages per question | 5 |
| Command | `uv run python scripts/benchmark_grounding.py --limit 8 --model sonnet` |

| | plain | grounded |
|---|---|---|
| right book | **8 of 8** | 5 of 8 |
| right passage | not asked | 5 of 8 |

So the closed-book exam went perfectly, and the open-book exam did worse. Here's how each question went:

| id | Question | plain | grounded |
|---|---|---|---|
| 1 | Where is Odysseus when the poem begins…? | right, Book 1 | right, cites `odyssey 1:5` |
| 6 | What did Aeolus give Odysseus…? | right, Book 10 | right, cites `odyssey 10:2`, and notes that the passages stop before the bag is opened |
| 11 | How does Odysseus … listen to the Sirens…? | right, Book 12 | right, cites `odyssey 12:5`, and notes it's Circe's advice, not the scene itself |
| 16 | Who recognises Odysseus by the scar…? | right, Book 19 | right, cites `odyssey 19:29` |
| 21 | Why is Achilles angry with Agamemnon? | right, Book 1 | *The passages don't say why* |
| 26 | Which goddess does Diomedes wound…? | right, Book 5 | right, cites `iliad 5:29` |
| 31 | How does Hera take Zeus's mind off the battle? | right, Book 14 | *The passages only hint at it* (cites a later mention in Book 15) |
| 36 | How does Athena deceive Hector…? | right, Book 22 | *The passages don't say* |

The plain answers weren't just the right book. I read all eight and they're correct, down to details like Athena taking the form of Hector's brother Deiphobus.

### ❓ Step 4: why did the grounded run lose three?

The search lost them, before Claude read a word. We can check that without calling any model: for the same 8 questions, was a right passage among the 5 that `search` returned?

👉 **Predict first:** for how many of the 8?

In [ ]:
from homer_ai.retrieval import first_hit_rank

# the 8 questions that --limit 8 picks: every fifth one
sample = [questions[int(i * len(questions) / 8)] for i in range(8)]
on_the_table = 0
for q in sample:
    rank = first_hit_rank([p for p, _ in search(q["question"], k=5)], q)
    on_the_table += rank is not None
    where = f"rank {rank}" if rank else "not in the 5"
    print(f"{q['id']:>2}  {where:14} {q['question']}")
print()
print(f"a right passage was on the table for {on_the_table} of {len(sample)}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

5 of 8, and they're the same five that the grounded run got right. For questions 21, 31 and 36 the search brought five passages that don't hold the answer, and Claude, told to answer only from them, said so.

So the answer step was as good as it could be: 5 of 5 when the passage was there. It can't do better than the search that feeds it. That's why 09_02 spent so long on recall@5. If you want better grounded answers, the search is usually the first place to look.

</details>

### ❓ Step 5: so the open book made things worse. Was this chapter pointless?

On this test, with this text, the closed book wins. I'd rather show you that than a benchmark picked to flatter the method. But look at what each side really gave us.

**The plain run was right because Claude has read Homer.** The Iliad and the Odyssey are among the most written-about texts there are. A model trained on the internet knows them the way you know your own street. Our test asked it about its home ground.

**The grounded run gave us things the plain run can't.**

- A line you can check. `odyssey 19:29` is a paragraph in a file. *Book 19* from memory is a claim.
- An honest *I don't know*. Three times the passages didn't hold the answer, and three times Claude said so. It didn't invent one. On a question the model really doesn't know, that's the difference between a gap and a confident mistake.
- The exact wording of *our* copy: Butler's *Noman*, not the *Nobody* of other translations.

**And most text isn't Homer.** Your team's handbook, last week's contract, the ticket history of your product: no model has read those. For them there is no closed-book option. The model either gets the page, or it guesses.

*This is the question I get asked first on real projects: where did that answer come from? A system that can point at the paragraph gets trusted. One that can't gets switched off after its first wrong answer, however good it was on average.*

One caution about the numbers. Eight questions is a tiny sample: one question is 12.5 points. This run shows the pattern. It measures nothing precisely.

### ❓ Step 6: what would I change first?

The search, since it's what lost the three. In the order I'd try things:

1. **Let the model search for itself.** In our benchmark the search ran once, with the question as typed. Through the MCP tool of 09_03, Claude can read the result, see that it's thin, and search again with other words. The tool's description tells it to. I tried that on question 10, *What does the ghost of Achilles say about being dead?*, which our one-shot search misses (it's in the list of 09_02). With the server connected to Claude Code (24 November 2026, `sonnet`), Claude called `search_homer` twice. The first call, with its own words for the scene, brought the paragraphs around the answer. The second asked for *rather be alive on earth as a paid servant to a poor man than rule over all the dead*, and got `odyssey 11:42`, which it quoted. It already knew roughly what the line says, and used that to find where it is. That's one question, so it's an example and not a score.
2. **Don't force the book closed.** We told Claude to answer *only* from the passages. In a real assistant you might let it say: the passages don't cover this; from memory, the answer is such-and-such, unverified.
3. **Better retrieval.** The hybrid of 09_02, a reranker, or smaller passages. Each one is a change you can score on the 40 questions before you believe in it.

### ❓ Where does this go in a real company?

Swap Homer for the company's own text, and every part of this chapter has a counterpart. The parts are the same. What grows is the care each one needs.

| In this chapter | In a company | What gets harder |
|---|---|---|
| 48 text files that never change | handbooks, contracts, tickets, wiki pages, PDFs | getting clean text out of them, and noticing when a document changes or is withdrawn |
| one paragraph, or 5 verses | passages cut along headings, clauses or table rows | a bad cut hides the answer; tables and lists need their own rules |
| `odyssey 9:24` | document, version, page or clause | the citation has to open the right version of the right file |
| BM25 over 2,140 passages | **hybrid search** over millions, often with **reranking** | speed, cost, and keeping the index up to date |
| everyone may read Homer | each person may read only some documents | the search must apply those permissions before anything reaches the model |
| 40 questions I wrote | hundreds of real questions from the people who do the job, with answers they signed off | someone has to own that set and keep it current |
| `search_homer` over MCP | the same kind of tool, behind a login | who called it, with what, and what came back: all of it logged |
| *the passages don't say* | the same sentence, plus where to ask instead | deciding when the system should refuse to answer |

If I had to pick the two rows that decide whether such a project works, they'd be the labelled questions and the permissions. The first tells you whether it's any good. The second decides whether you're allowed to switch it on.

## Questions learners ask

These are real questions that came up while we were building this project, with short answers.

### ❓ Could I run the benchmark with another model?

Yes: `--model` takes any name your Claude Code accepts, and `--limit 40` runs every question (80 calls). A weaker model will likely do worse on the plain run, and that's where grounding starts to pay. To use a different program altogether, `ask_claude` in the script is the one function to replace.

### ❓ Why not let a model judge whether each answer is right?

You can, and people do; it's called an LLM judge. It reads wording, which our two scores don't. But then the judge needs checking too. A fixed `SOURCE:` line and a labelled range are crude, and you can see exactly what they measure. I start with the crude score and read the answers myself.

### ❓ Does grounding stop a model from making things up?

It makes it less likely and easier to catch. A model can still misread a passage, or blend two. What changes is that you can check: the reference is right there. In the run above it also turned three would-be guesses into three plain *the passages don't say*.

### ❓ Is this what people mean by RAG?

Yes. Retrieve (the search), augment (paste the passages into the prompt), generate (the model writes). The name makes it sound like one technique. As you've seen, it's a pipeline of ordinary parts, and each part can be measured by itself.

## 5. Recap

- The passages reach the model as text in the prompt, each with its reference, plus an instruction to answer only from them and to say when they don't hold the answer.
- In one small run (8 questions), Claude named the right book 8 times from memory and 5 times from our passages. Homer is home ground for a big model.
- The 3 losses were the search's: no right passage among the 5. When the passage was there, the answer and its citation were right every time.
- What grounding adds is a line you can check and an honest *I don't know*. On text no model has read, it's the only way to get an answer at all.
- In a company the same parts return, with more care: clean text, good cuts, permissions, and above all a set of labelled questions.

### ❓ What did this chapter build?

- Passages of both poems that know their book and line.
- BM25 from scratch, with every number checked by hand on three passages.
- A set of 40 labelled questions, and two scores to read it with.
- A second MCP tool, and the settings to use it from five AI apps.
- A with-and-without test that told us something we didn't expect.

**Next:** [10_00: Your own domain](../10_advanced_your_domain/10_00_chapter_overview.ipynb), where the text is a company's own. Or go [back to the course map](../00_start_here/00_00_course_map.ipynb).